In [1]:
# ==========================================================
# Table 1: Feature Selection Scores (FINAL - RESEARCH GRADE)
# ==========================================================

!pip install dcor pywavelets

import os
import numpy as np
import pandas as pd
from scipy import stats, signal
import dcor
import pywt
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# ==========================================================
# 0. Load Data
# ==========================================================
df = pd.read_csv("Metals_Price.csv", parse_dates=["Date"])
df.set_index("Date", inplace=True)

ret = np.log(df / df.shift(1)).dropna()

target = ret["Aluminium"]
features = ["Copper", "Nickel", "Zinc", "Tin", "Lead"]
short = ["Cu", "Ni", "Zn", "Sn", "Pb"]

# ==========================================================
# Helper: Z-score normalization per method
# ==========================================================
def zscore_normalise(raw):
    vals = np.array(list(raw.values()))
    mean = vals.mean()
    std = vals.std() + 1e-12
    return {k: (v - mean)/std for k, v in raw.items()}

# ==========================================================
# 1. Pearson (with significance)
# ==========================================================
def calc_pearson():
    raw = {}
    for f in features:
        r, p = stats.pearsonr(target, ret[f])
        raw[f] = abs(r) if p < 0.05 else 0.0
    return raw

# ==========================================================
# 2. Spearman
# ==========================================================
def calc_spearman():
    raw = {}
    for f in features:
        r, p = stats.spearmanr(target, ret[f])
        raw[f] = abs(r) if p < 0.05 else 0.0
    return raw

# ==========================================================
# 3. Kendall
# ==========================================================
def calc_kendall():
    raw = {}
    for f in features:
        r, p = stats.kendalltau(target, ret[f])
        raw[f] = abs(r) if p < 0.05 else 0.0
    return raw

# ==========================================================
# 4. Distance Correlation
# ==========================================================
def calc_dcor():
    raw = {}
    for f in features:
        raw[f] = dcor.distance_correlation(target.values, ret[f].values)
    return raw

# ==========================================================
# 5. HSIC (FIXED BANDWIDTH)
# ==========================================================
def rbf_kernel(x):
    x = x.reshape(-1,1)
    sq = np.sum((x[:,None]-x[None,:])**2, axis=-1)
    sigma = np.std(x) * 0.5 + 1e-6
    return np.exp(-sq/(2*sigma**2))

def hsic(x,y):
    n = len(x)
    K = rbf_kernel(x)
    L = rbf_kernel(y)
    H = np.eye(n) - np.ones((n,n))/n
    return np.trace(K@H@L@H)/(n-1)**2

def calc_hsic():
    raw = {}
    for f in features:
        raw[f] = hsic(target.values, ret[f].values)
    return raw

# ==========================================================
# 6. Mutual Information
# ==========================================================
def calc_mi():
    raw = {}
    for f in features:
        vals = []
        for rs in range(5):
            mi = mutual_info_regression(
                ret[f].values.reshape(-1,1),
                target.values,
                n_neighbors=7,
                random_state=rs
            )[0]
            vals.append(mi)
        raw[f] = np.mean(vals)
    return raw

# ==========================================================
# 7. Transfer Entropy (LOG-SCALED)
# ==========================================================
def transfer_entropy(x, y, lag=1):
    bins = int(np.sqrt(len(x)))

    y_now = y[lag:]
    y_past = y[:-lag]
    x_past = x[:-lag]

    def digitize(a):
        return np.digitize(a, np.histogram_bin_edges(a, bins=bins))

    y_now, y_past, x_past = map(digitize, [y_now, y_past, x_past])

    def entropy(*args):
        joint = np.column_stack(args)
        _, counts = np.unique(joint, axis=0, return_counts=True)
        p = counts / counts.sum()
        return -np.sum(p * np.log2(p + 1e-12))

    te = (
        entropy(y_now, y_past)
        + entropy(y_past, x_past)
        - entropy(y_now, y_past, x_past)
        - entropy(y_past)
    )
    return np.log1p(max(te,0))   # 🔥 FIX

def calc_te():
    raw = {}
    for f in features:
        raw[f] = transfer_entropy(ret[f].values, target.values)
    return raw

# ==========================================================
# 8. Tail Dependence
# ==========================================================
def tail_dependence(x,y,q=0.05):
    n=len(x)
    u = stats.rankdata(x)/(n+1)
    v = stats.rankdata(y)/(n+1)

    lower = np.mean(v[u<=q]<=q) if np.sum(u<=q)>0 else 0
    upper = np.mean(v[u>=1-q]>=1-q) if np.sum(u>=1-q)>0 else 0
    return (lower+upper)/2

def calc_tail():
    raw={}
    for f in features:
        raw[f]=tail_dependence(target.values,ret[f].values)
    return raw

# ==========================================================
# 9. Wavelet (Correct D5)
# ==========================================================
def wavelet_corr(x,y,level=5):
    x=np.asarray(x).copy()
    y=np.asarray(y).copy()

    max_level=pywt.dwt_max_level(len(x),'db4')
    level=min(level,max_level)

    cx=pywt.wavedec(x,'db4',level=level)
    cy=pywt.wavedec(y,'db4',level=level)

    dx=cx[level]
    dy=cy[level]

    r,_=stats.pearsonr(dx,dy)
    return abs(r)

def calc_wavelet():
    raw={}
    for f in features:
        raw[f]=wavelet_corr(target.values,ret[f].values)
    return raw

# ==========================================================
# 10. Spectral Coherence (LOW-FREQ BAND)
# ==========================================================
def spectral_coherence(x,y):
    f,Cxy = signal.coherence(x,y,nperseg=min(256,len(x)//4))
    mask = (f>0.01)&(f<0.1)
    return np.mean(Cxy[mask])

def calc_spectral():
    raw={}
    for f in features:
        raw[f]=spectral_coherence(target.values,ret[f].values)
    return raw

# ==========================================================
# 11. Lagged Partial (RIDGE)
# ==========================================================
def partial_corr(target,feature,others,max_lag=5):
    scaler=StandardScaler()
    best=0
    lam=1e-3

    for lag in range(1,max_lag+1):
        y=target.iloc[lag:].values
        x=feature.iloc[:-lag].values
        Z=others.iloc[:-lag].values

        Z=scaler.fit_transform(Z)
        Zc=np.column_stack([Z,np.ones(len(Z))])

        try:
            beta_y=np.linalg.inv(Zc.T@Zc + lam*np.eye(Zc.shape[1]))@Zc.T@y
            beta_x=np.linalg.inv(Zc.T@Zc + lam*np.eye(Zc.shape[1]))@Zc.T@x

            ry=y-Zc@beta_y
            rx=x-Zc@beta_x

            r,_=stats.pearsonr(ry,rx)
            best=max(best,abs(r))
        except:
            pass

    return best

def calc_partial():
    raw={}
    for f in features:
        others=[c for c in features if c!=f]
        raw[f]=partial_corr(target,ret[f],ret[others])
    return raw

# ==========================================================
# COMPUTE ALL METHODS
# ==========================================================
methods_raw = {
    "Pearson": calc_pearson(),
    "Spearman": calc_spearman(),
    "Kendall": calc_kendall(),
    "dCor": calc_dcor(),
    "HSIC": calc_hsic(),
    "MI": calc_mi(),
    "TE": calc_te(),
    "Tail": calc_tail(),
    "Wavelet": calc_wavelet(),
    "Spectral": calc_spectral(),
    "LagPartial": calc_partial(),
}

# ==========================================================
# Z-SCORE NORMALIZATION (CRITICAL FIX)
# ==========================================================
methods = {k: zscore_normalise(v) for k,v in methods_raw.items()}

# ==========================================================
# BUILD TABLE
# ==========================================================
rows=[]
for m,scores in methods.items():
    row={"Method":m}
    for f,s in zip(features,short):
        row[s]=scores[f]
    rows.append(row)

table=pd.DataFrame(rows).set_index("Method")

means=table.mean()
ranks=means.rank(ascending=False).astype(int)

table.loc["Mean"]=means
table.loc["Rank"]=ranks

# ==========================================================
# SAVE RESULTS
# ==========================================================

table.to_csv("feature_scores.csv")
means.to_csv("mean_scores.csv")

# ==========================================================
# PRINT
# ==========================================================
print("\n"+"="*75)
print(" FINAL FEATURE SELECTION TABLE (RESEARCH-GRADE)")
print("="*75)
print(table.round(3).to_string())
print("="*75)

print("\nSaved in: feature_selection_final/")


 FINAL FEATURE SELECTION TABLE (RESEARCH-GRADE)
               Cu     Ni     Zn     Sn     Pb
Method                                       
Pearson     1.166 -0.022  0.831 -1.695 -0.280
Spearman    1.302 -0.208  0.862 -1.528 -0.428
Kendall     1.313 -0.238  0.873 -1.503 -0.446
dCor        1.297 -0.211  0.870 -1.527 -0.430
HSIC        1.352 -0.309  0.918 -1.376 -0.586
MI          1.426 -0.332  0.794 -1.416 -0.471
TE          0.142 -1.780  1.291 -0.031  0.378
Tail        1.177 -0.285  1.061 -1.477 -0.477
Wavelet     1.171  0.156  0.699 -1.743 -0.283
Spectral    0.834 -0.433  1.437 -1.346 -0.492
LagPartial  1.154  0.921 -1.019 -1.310  0.254
Mean        1.121 -0.249  0.783 -1.359 -0.296
Rank        1.000  3.000  2.000  5.000  4.000

Saved in: feature_selection_final/
